In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start

repo_root = _find_repo_root(Path.cwd())
workspace_root = repo_root.parent
candidate_src_dirs = [
    repo_root / "src",
    workspace_root / "abstractgraph" / "src",
    workspace_root / "abstractgraph-ml" / "src",
    workspace_root / "abstractgraph-generative" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
from abstractgraph.graphs import AbstractGraph
from abstractgraph.display import display, display_graph, display_mappings, display_decomposition_graph, decomposition_to_graph
from abstractgraph.operators import *

def draw(graph, df, nbits=10):
    display_decomposition_graph(decomposition_to_graph(df))
    ag = AbstractGraph(graph=graph).create_default_image_node().update()
    ag = df(ag)
    ag.update()
    #print(ag)
    display(ag, size=(12,6))
    display_mappings(ag)

In [ ]:
USE_EXAMPLE = 'chem'

if USE_EXAMPLE == 'random':
    from coco_grape.utils.artificial_graph_constructor import RandomGraphConstructor
    graph = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4).sample(1)
else:
    smi = 'CC1NC(OC2CC(O)C3(CO)C4C(O)CC5(C)C(C6CNC(=O)C6)CCC5(O)C4CCC3(O)C2)C(O)C(O)C1O'
    from coco_grape.data_graphicalizer.mol.molecular_graphicalizer import SmilesMolecularGraphicalizer
    graph = SmilesMolecularGraphicalizer().transform([smi])[0]
    from coco_grape.visualizer.mol_display import draw_molecules
    draw_molecules([graph], n_graphs_per_line=1)

print(graph)

display_graph(graph)

In [ ]:
cycle_df = compose(filter_by_number_of_nodes(number_of_nodes=(5,6)), cycle())
cycle_with_N_df = compose(filter_by_node_label(must_have_one_of=['N']), cycle())
pairs_df = compose_product(binary_combination(distance=(1,4)), cycle_df, cycle_with_N_df)
df = compose(intersection_edges(), pairs_df)
draw(graph, df)

In [ ]:
from abstractgraph.xml import register_from_module, operator_to_xml_string, operator_from_xml_string
import abstractgraph.operators as qg_ops
register_from_module(qg_ops)
xml_df = operator_to_xml_string(df, pretty=True)
print(xml_df)


In [ ]:
df_rt = operator_from_xml_string(xml_df)
draw(graph, df)

---